In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder


In [ ]:
data = pd.read_csv("../data/survey_results.csv")

data.head()

### 1. Data Cleaning 

In [ ]:
data.duplicated().sum()

In [ ]:
data[data.duplicated(keep= "first")]

In [ ]:
data.drop_duplicates(inplace= True)

In [ ]:
data.duplicated().sum()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data= data, x = "age")
plt.tight_layout()
plt.show()

In [ ]:
data["age"].describe()

In [ ]:
thres = data["age"].quantile(0.9997)

thres

In [ ]:
df = data[data["age"] < 100].copy()

df.head()


In [ ]:
df.isna().sum()

In [ ]:
df[df["income_levels"].isna()]["occupation"].value_counts()

In [ ]:
df["income_levels"].unique()

In [ ]:
df["income_levels"] = df["income_levels"].fillna("not reported")

In [ ]:
df["income_levels"].unique()

In [ ]:
df["consume_frequency(weekly)"].unique()

In [ ]:
df["purchase_channel"].unique()

In [ ]:
df["consume_frequency(weekly)"].mode()[0]

In [ ]:
df["consume_frequency(weekly)"] = df["consume_frequency(weekly)"].fillna(df["consume_frequency(weekly)"].mode()[0])

In [ ]:
df["purchase_channel"].mode()[0]

In [ ]:
df["purchase_channel"] = df["purchase_channel"].fillna(df["purchase_channel"].mode()[0])

In [ ]:
df.isna().sum()

In [ ]:
df["zone"].unique()

In [ ]:
len(df[df["zone"] == "Metor"])

In [ ]:
df["current_brand"].unique()

In [ ]:
df["zone"] = df["zone"].replace({
    'urbna': 'Urban',
    'Metor': 'Metro'

})

In [ ]:
df["zone"].unique()

In [ ]:
df["current_brand"] = df["current_brand"].replace({
    'Establishd': 'Established',
    'newcomer': 'Newcomer'
})

In [ ]:
df["current_brand"].unique()

### 2. Feature Engineering

In [ ]:
bins = [18, 26, 36, 46, 56, 71, 101]

labels = ["18-25", "26-35", "36-45", "46-55", "56-70", "70+"]

df['age_group'] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    right=False
)


In [ ]:
df.head(2)

In [ ]:
df["age_group"].unique()

In [ ]:
df["consume_frequency(weekly)"].unique()

In [ ]:
df["consume_frequency(weekly)"] = df["consume_frequency(weekly)"].map({
    '0-2 times': 1,
    '3-4 times': 2,
    '5-7 times': 3
})


In [ ]:
df["awareness_of_other_brands"].unique()

In [ ]:
df["awareness_of_other_brands"] = df["awareness_of_other_brands"].map({
    '0 to 1': 1,
    '2 to 4': 2,
    'above 4': 3
})

df.head(2)

In [ ]:
df["cf_ab_score"] = (
    df["consume_frequency(weekly)"] / (df["consume_frequency(weekly)"] + df["awareness_of_other_brands"])
).round(2)

In [ ]:
df["cf_ab_score"].describe()

In [ ]:
df["zone"].unique()

In [ ]:
df["zone"] = df["zone"].map({
    'Rural': 1,
    'Semi-Urban': 2,
    'Urban': 3,
    'Metro': 4
})

In [ ]:
df["income_levels"].unique()

In [ ]:
df["income_levels"] = df["income_levels"].map({
    'not reported': 0,
    '<10L': 1,
    '10L - 15L': 2,
    '16L - 25L': 3,
    '26L - 35L': 4,
    '> 35L': 5
})

In [ ]:
df.head(2)

In [ ]:
df['zas_score'] = df["zone"] * df["income_levels"]

df.head(2)

In [ ]:
len(df["zas_score"].unique())

In [ ]:
df['bsi'] = np.where(
    (df["current_brand"] != "Established") & 
    (df["reasons_for_choosing_brands"].isin(["Price", "Quality"])),
    1, 0
)

df.head(2)

In [ ]:
df.shape

In [ ]:
pd.crosstab(df["age_group"], df["occupation"])

In [ ]:
mask = (df["age_group"] =="56-70") & (df["occupation"] == "Student")

df = df[~mask]

pd.crosstab(df["age_group"], df["occupation"])

In [ ]:
df.shape

In [ ]:
df["bsi"].value_counts()

### 3. Predictive Modeling

In [ ]:
df.columns

In [ ]:
X = df.drop(columns=["respondent_id", "price_range", "age"])
y = df["price_range"]

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state= 42)

In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
X["age_group"].value_counts()

In [ ]:
X["health_concerns"].value_counts()

In [ ]:
X["preferable_consumption_size"].value_counts()

In [ ]:
X.select_dtypes('object').columns

In [ ]:
X["income_levels"].value_counts()

In [ ]:
X.select_dtypes('object').columns

In [ ]:
ct = ColumnTransformer(transformers=[

    ('oe', OrdinalEncoder(categories=[
        ["18-25", "26-35", "36-45", "46-55", "56-70", "70+"],
        ["Low (Not very concerned)", 
         "Medium (Moderately health-conscious)", 
         "High (Very health-conscious)"],
        ["Small (250 ml)", "Medium (500 ml)", "Large (1 L)"]
    ]),
     ["age_group", "health_concerns", "preferable_consumption_size"]
    ),

    ('one', OneHotEncoder(drop= 'first'),
     ["gender", "occupation", "current_brand", "reasons_for_choosing_brands", "flavor_preference", "purchase_channel", "packaging_preference", "typical_consumption_situations"]
    )
], remainder='passthrough', verbose_feature_names_out= False)

In [ ]:
X_trian_transformed = pd.DataFrame(ct.fit_transform(X_train), columns= ct.get_feature_names_out())
X_test_transformed = pd.DataFrame(ct.transform(X_test), columns= ct.get_feature_names_out())

In [ ]:
X_trian_transformed.head()

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

gnb.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = gnb.predict(X_test_transformed)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter= 1000)
logreg.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = logreg.predict(X_test_transformed)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.svm import SVC

svc = SVC()
svc.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = svc.predict(X_test_transformed)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
rfc.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = rfc.predict(X_test_transformed)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier()
xgb.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = xgb.predict(X_test_transformed)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier()
lgbm.fit(X_trian_transformed, y_train)

In [ ]:
y_pred = lgbm.predict(X_test_transformed)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
import mlflow
from sklearn.pipeline import Pipeline
import pandas as pd
import mlflow.sklearn
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)



mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("Beverage Price Prediction")

models = {
    "Gaussian Naive Bayes": GaussianNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "Support Vector Machine(SVM)": SVC(),
    "RandomForest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(),
    "LightGBM": LGBMClassifier()
}


for model_name, model in models.items():          

    with mlflow.start_run(run_name=model_name):   

        pipe = Pipeline(steps=[
            ('processing', ct),
            ('models', model)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        # Metrics
        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall    = recall_score(y_test, y_pred, average="weighted")
        f1        = f1_score(y_test, y_pred, average="weighted")

        mlflow.log_metric("accuracy",  accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall",    recall)
        mlflow.log_metric("f1_score",  f1)

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure()
        sns.heatmap(cm, annot=True, fmt="d", cmap="RdBu")
        plt.title(f"{model_name} Confusion Matrix")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        cm_path = f"{model_name}_confusion_matrix.png"
        plt.savefig(cm_path)
        plt.close()
        mlflow.log_artifact(cm_path)

        # Classification Report
        report = classification_report(y_test, y_pred, output_dict=True)
        report_df = pd.DataFrame(report).transpose()
        report_path = f"{model_name}_classification_report.csv"
        report_df.to_csv(report_path)
        mlflow.log_artifact(report_path)

        #Log the model
        mlflow.sklearn.log_model(
            pipe[1],                        
            name=f"{model_name}"    

        )

        print(f"{model_name} logged successfully.")
